## Dimensional Bronze Layer - Streaming Ingestion

Ingesting dimensional data from raw landing zone using Auto Loader:
* **brz_brands** - Brand master data
* **brz_category** - Product categories
* **brz_products** - Product catalog
* **brz_customers** - Customer master data
* **brz_date** - Calendar dimension



**Key Features**:
- Auto Loader with explicit schemas
- Unity Catalog volumes for checkpoints
- Metadata tracking (_ingested_at, _source_file)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DateType, TimestampType, StringType, IntegerType, LongType, DoubleType

# Defining paths
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
catalog_name = dbutils.widgets.get("catalog_name")
checkpoint_base = f"/Volumes/{catalog_name}/raw/checkpoints"
landing_base = f"/Volumes/{catalog_name}/raw/raw_landing"

print(f"Catalog: {catalog_name}")
print(f"Checkpoint base: {checkpoint_base}")
print(f"Landing base: {landing_base}")

# Defining schemas for each data file
schemas = {
  "brands": StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True),
]),
  "category": StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
]),
"products": StructType([
    StructField("product_id", LongType(), False), 
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),
    StructField("length_cm", StringType(), True),
    StructField("width_cm", DoubleType(), True),
    StructField("height_cm", DoubleType(), True),  
    StructField("rating_count", IntegerType(), True),
]),
"customers": StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
]),
"date": StructType([
    StructField("date", DateType(), True),
    StructField("year", IntegerType(), True),
    StructField("day_name", StringType(), True),
    StructField("quarter", IntegerType(), True),
    StructField("week_of_year", IntegerType(), True),
])
}

In [0]:
# Adding metadata for each table and writting into Delta Tables
for table_name, schema in schemas.items():
    print(f"Processing category: {table_name}")

    checkpoint_data = f"{checkpoint_base}/brz_{table_name}/data"
    checkpoint_schema = f"{checkpoint_base}/brz_{table_name}/schema"
    source_path = f"{landing_base}/{table_name}/"

    # Read data from landing zone using Auto Loader
    df_stream = (spark.readStream
                 .format("cloudFiles")
                 .option("cloudFiles.format", "csv")
                 .option("cloudFiles.schemaLocation", checkpoint_schema)
                 .option("cloudFiles.schemaEvolutionMode", "rescue")
                 .option("header", "true")
                 .schema(schema)
                 .load(source_path)
    )

    # Adding metadata 
    df_final = df_stream.select(
        "*",
        F.current_timestamp().alias("_ingested_at"),
        F.col("_metadata.file_path").alias("_source_file")
    )

    (df_final.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_data)
            .option("mergeSchema", "true")
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(f"{catalog_name}.bronze.brz_{table_name}")
            .awaitTermination()
    )
    print(f"Completed: {table_name}")

print(f"All dimensional tables processed successfully!")

In [0]:
# Check row counts for all bronze tables
tables = ['brz_brands', 'brz_category', 'brz_products', 'brz_customers', 'brz_date']

for table in tables:
    try:
        count = spark.table(f"{catalog_name}.bronze.{table}").count()
        print(f"{table:20} : {count:,} rows")
    except Exception as e:
        print(f"{table:20} : Table not found!")